In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
from scipy import stats
from sklearn.preprocessing import StandardScaler

# Ejemplo NN con output numerico continuo

In [ ]:
# ─────────────────────────────────────────
# 1. DATOS SIMULADOS (precios de casas)
# ─────────────────────────────────────────
np.random.seed(42)
n = 500

# Features: metros², habitaciones, distancia al centro
metros     = np.random.uniform(40, 300, n)
habitaciones = np.random.randint(1, 6, n).astype(float)
distancia  = np.random.uniform(1, 50, n)

# Target: precio en USD (distribución log-normal, típico en precios reales)
precio = (metros * 1500) + (habitaciones * 20000) - (distancia * 3000) + np.random.normal(0, 30000, n)
precio = np.abs(precio)  # no queremos negativos

X = np.column_stack([metros, habitaciones, distancia])
y = precio.reshape(-1, 1)

print(f"Rango de precios: ${y.min():,.0f} - ${y.max():,.0f}")
print(f"Media: ${y.mean():,.0f}\n")


# ─────────────────────────────────────────
# 2. PREPROCESAMIENTO
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar features (siempre recomendado)
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

# en el caso output continuo, aplicamos log por motivos de estabilidad numerica y manejo de outliers.

# ✅ Aplicar log1p al target -> gradientes más estables y manejo de outliers
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# Convertir a tensores
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train_log, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_log,  dtype=torch.float32)


# ─────────────────────────────────────────
# 3. MODELO
# ─────────────────────────────────────────
class RedPrecios(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Sin activación final → regresión libre
        )

    def forward(self, x):
        return self.red(x)

modelo = RedPrecios()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)
loss_fn = nn.MSELoss()  # opera sobre valores log → estable


# ─────────────────────────────────────────
# 4. ENTRENAMIENTO
# ─────────────────────────────────────────
epochs = 2000

for epoch in range(epochs):
    modelo.train()
    optimizer.zero_grad()

    pred_log = modelo(X_train_t)         # predice en escala log
    loss = loss_fn(pred_log, y_train_t)  # MSE sobre log → gradientes estables

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1:3d} | Loss (log-scale): {loss.item():.4f}")


# ─────────────────────────────────────────
# 5. EVALUACIÓN
# ─────────────────────────────────────────
modelo.eval()
with torch.no_grad():
    pred_log = modelo(X_test_t).numpy()

# ✅ Revertir log1p → precios reales
pred_real = np.expm1(pred_log)
y_real    = np.expm1(y_test_log)

# MAE en dólares reales
mae = np.abs(pred_real - y_real).mean()
print(f"\nMAE en precios reales: ${mae:,.0f}")

# Ver algunas predicciones vs reales
print("\nEjemplos (real vs predicho):")
print(f"{'Real':>15} | {'Predicho':>15} | {'Error':>12}")
print("-" * 47)
for r, p in zip(y_real[:5], pred_real[:5]):
    error = abs(r[0] - p[0])
    print(f"${r[0]:>13,.0f} | ${p[0]:>13,.0f} | ${error:>10,.0f}")

Rango de precios: $428 - $502,222
Media: $236,282

Epoch 200 | Loss (log-scale): 4.2805
Epoch 400 | Loss (log-scale): 2.3400
Epoch 600 | Loss (log-scale): 1.1234
Epoch 800 | Loss (log-scale): 0.5267
Epoch 1000 | Loss (log-scale): 0.2931
Epoch 1200 | Loss (log-scale): 0.2257
Epoch 1400 | Loss (log-scale): 0.2085
Epoch 1600 | Loss (log-scale): 0.2007
Epoch 1800 | Loss (log-scale): 0.1958
Epoch 2000 | Loss (log-scale): 0.1919

MAE en precios reales: $36,542

Ejemplos (real vs predicho):
           Real |        Predicho |        Error
-----------------------------------------------
$      259,931 | $      298,941 | $    39,010
$      348,529 | $      372,353 | $    23,824
$       84,687 | $       90,452 | $     5,766
$      118,622 | $      161,869 | $    43,246
$      466,362 | $      557,738 | $    91,375


# Ejemplo de NN con output numerico discreto

In [11]:
# ─────────────────────────────────────────
# 1. DATOS SIMULADOS (número de pedidos)
# ─────────────────────────────────────────
np.random.seed(42)
n = 500

# Features: antigüedad del cliente (días), gasto histórico, edad
antiguedad = np.random.uniform(1, 1000, n)
gasto      = np.random.uniform(10, 5000, n)
edad       = np.random.uniform(18, 70, n)

# Target: número de pedidos (entero, distribución de Poisson típica en conteos)
# Los conteos suelen ser bajos con algunos valores altos (skewed)
lambda_pedidos = 0.005 * antiguedad + 0.002 * gasto / 100
pedidos = np.random.poisson(lambda_pedidos).astype(float)

X = np.column_stack([antiguedad, gasto, edad])
y = pedidos.reshape(-1, 1)

print(f"Rango de pedidos: {int(y.min())} - {int(y.max())}")
print(f"Media: {y.mean():.2f}")
print(f"Distribución: {np.bincount(pedidos.astype(int))[:10]} (primeros 10 valores)\n")


# ─────────────────────────────────────────
# 2. PREPROCESAMIENTO
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar features (siempre)
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

# ✅ OPCIÓN RECOMENDADA: log1p también funciona muy bien para conteos
# porque los conteos también suelen ser skewed (muchos 0s, pocos valores altos)

# los pedidos tienen un problema: muchos clientes con 0-5 pedidos y algunos con 50+, 
# lo que crea una cola larga. Sin el log1p, la red habría sobreajustado esos casos extremos.
# en casos continuo aplicamos log solo por la distribucion, por no ponderar demasiado casos extremos.
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# Convertir a tensores
X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_sc,  dtype=torch.float32)
y_train_t = torch.tensor(y_train_log, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_log,  dtype=torch.float32)


# ─────────────────────────────────────────
# 3. MODELO
# ─────────────────────────────────────────
class RedPedidos(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
            # ✅ Sin activación final: la red predice en escala log (puede ser negativo)
            # El redondeo a entero lo hacemos DESPUÉS de revertir con expm1
        )

    def forward(self, x):
        return self.red(x)

modelo = RedPedidos()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)
loss_fn = nn.MSELoss()


# ─────────────────────────────────────────
# 4. ENTRENAMIENTO
# ─────────────────────────────────────────
epochs = 500

for epoch in range(epochs):
    modelo.train()
    optimizer.zero_grad()

    pred_log = modelo(X_train_t)
    loss = loss_fn(pred_log, y_train_t)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d} | Loss (log-scale): {loss.item():.4f}")


# ─────────────────────────────────────────
# 5. EVALUACIÓN
# ─────────────────────────────────────────
modelo.eval()
with torch.no_grad():
    pred_log = modelo(X_test_t).numpy()

# ✅ Revertir log1p y redondear al entero más cercano
pred_continuo = np.expm1(pred_log)
pred_entero   = np.round(pred_continuo).clip(0).astype(int)  # clip(0) → no negativos
y_real        = y_test.astype(int)

# Métricas
mae  = np.abs(pred_entero - y_real).mean()
rmse = np.sqrt(((pred_entero - y_real) ** 2).mean())
print(f"\nMAE:  {mae:.2f} pedidos")
print(f"RMSE: {rmse:.2f} pedidos")

# Ver algunas predicciones vs reales
print("\nEjemplos (real vs predicho):")
print(f"{'Real':>8} | {'Pred (float)':>14} | {'Pred (int)':>11} | {'Error':>6}")
print("-" * 40)
for r, pc, pi in zip(y_real[:8], pred_continuo[:8], pred_entero[:8]):
    error = abs(r[0] - pi[0])
    print(f"{r[0]:>8} | {pc[0]:>14.3f} | {pi[0]:>11} | {error:>6}")

Rango de pedidos: 0 - 10
Media: 2.47
Distribución: [ 95 108  98  62  51  24  31  17  10   3] (primeros 10 valores)

Epoch 100 | Loss (log-scale): 0.2417
Epoch 200 | Loss (log-scale): 0.2108
Epoch 300 | Loss (log-scale): 0.1963
Epoch 400 | Loss (log-scale): 0.1908
Epoch 500 | Loss (log-scale): 0.1880

MAE:  1.29 pedidos
RMSE: 1.80 pedidos

Ejemplos (real vs predicho):
    Real |   Pred (float) |  Pred (int) |  Error
----------------------------------------
       6 |          2.182 |           2 |      4
       1 |          3.030 |           3 |      2
       1 |          0.297 |           0 |      1
       2 |          0.919 |           1 |      1
       4 |          3.519 |           4 |      0
       1 |          1.535 |           2 |      1
       1 |          0.341 |           0 |      1
       2 |          1.039 |           1 |      1


# Ejemplo de NN con output numerico discreto pero CATEGORICO

In [12]:
# ─────────────────────────────────────────
# 1. DATOS SIMULADOS
# Predecir segmento de cliente: 0=bajo, 1=medio, 2=alto
# Aunque son números, NO tienen relación ordinal real para el modelo
# ─────────────────────────────────────────
np.random.seed(42)
n = 600

antiguedad = np.random.uniform(1, 1000, n)
gasto      = np.random.uniform(10, 5000, n)
edad       = np.random.uniform(18, 70, n)

# Asignar clases basadas en gasto (simulando segmentación real)
clases = np.where(gasto < 1000, 0, np.where(gasto < 3000, 1, 2))
# Añadir algo de ruido para que no sea trivial
ruido = np.random.randint(0, 3, n)
mask = np.random.rand(n) < 0.15        # 15% de muestras con ruido
clases[mask] = ruido[mask]

X = np.column_stack([antiguedad, gasto, edad])
y = clases  # ✅ enteros simples: 0, 1 o 2 — sin transformar, sin one-hot

print(f"Clases: {np.unique(y)}")
print(f"Distribución: {np.bincount(y)}\n")  # cuántos de cada clase


# ─────────────────────────────────────────
# 2. PREPROCESAMIENTO
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_sc,  dtype=torch.float32)

# ✅ Target como Long (entero 64-bit) — lo que exige CrossEntropyLoss
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)


# ─────────────────────────────────────────
# 3. MODELO
# ─────────────────────────────────────────
class RedClasificacion(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3)   # ✅ 3 neuronas de salida — una por clase
            # ❌ Sin softmax aquí: CrossEntropyLoss lo aplica internamente
        )

    def forward(self, x):
        return self.red(x)     # devuelve logits crudos: [2.1, -0.5, 1.3]

modelo = RedClasificacion()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)

# ✅ CrossEntropyLoss: hace softmax + log + NLL internamente
# Espera logits (no probabilidades) y clases como enteros
loss_fn = nn.CrossEntropyLoss()


# ─────────────────────────────────────────
# 4. ENTRENAMIENTO
# ─────────────────────────────────────────
epochs = 200

for epoch in range(epochs):
    modelo.train()
    optimizer.zero_grad()

    logits = modelo(X_train_t)          # shape: [batch, 3]
    loss = loss_fn(logits, y_train_t)   # compara logits con clases enteras

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")


# ─────────────────────────────────────────
# 5. EVALUACIÓN
# ─────────────────────────────────────────
modelo.eval()
with torch.no_grad():
    logits = modelo(X_test_t)               # logits crudos

    # Convertir logits → probabilidades (solo para interpretación)
    probs = torch.softmax(logits, dim=1)

    # ✅ La predicción final es el argmax (clase con mayor probabilidad)
    pred_clases = torch.argmax(probs, dim=1).numpy()

accuracy = (pred_clases == y_test).mean()
print(f"\nAccuracy: {accuracy:.2%}")

# Ver algunas predicciones con sus probabilidades
print("\nEjemplos (real vs predicho):")
print(f"{'Real':>6} | {'Pred':>6} | {'P(bajo)':>9} | {'P(medio)':>9} | {'P(alto)':>9}")
print("-" * 52)
for i in range(8):
    r  = y_test[i]
    p  = pred_clases[i]
    pb = probs[i][0].item()
    pm = probs[i][1].item()
    pa = probs[i][2].item()
    ok = "✅" if r == p else "❌"
    print(f"{r:>6} | {p:>6} | {pb:>9.3f} | {pm:>9.3f} | {pa:>9.3f}  {ok}")

Clases: [0 1 2]
Distribución: [140 226 234]

Epoch  50 | Loss: 0.6745
Epoch 100 | Loss: 0.5303
Epoch 150 | Loss: 0.4613
Epoch 200 | Loss: 0.4198

Accuracy: 87.50%

Ejemplos (real vs predicho):
  Real |   Pred |   P(bajo) |  P(medio) |   P(alto)
----------------------------------------------------
     2 |      2 |     0.070 |     0.060 |     0.870  ✅
     0 |      0 |     0.982 |     0.016 |     0.002  ✅
     2 |      2 |     0.075 |     0.017 |     0.907  ✅
     1 |      1 |     0.023 |     0.965 |     0.012  ✅
     1 |      1 |     0.040 |     0.752 |     0.208  ✅
     2 |      2 |     0.076 |     0.022 |     0.902  ✅
     0 |      0 |     0.977 |     0.017 |     0.006  ✅
     1 |      1 |     0.096 |     0.880 |     0.024  ✅
